# Train C, D, 0b, 0c -- Google Colab

## 1. Overview

Focused companion to `notebooks/train_evaluate_colab.ipynb`: trains,
calibrates, and evaluates only four experiments --

- **0b** (`exp_0b_plain_deep18_no_skip_shared_adapters_learned_balance`) --
  the depth/width-matched no-skip-connection control.
- **0c** (`exp_0c_custom_resnet18_no_zero_init_shared_adapters_learned_balance`) --
  Custom ResNet-18 with `zero_init_residual=false`.
- **C** (`exp_c_shared_adapters`) -- shared backbone + adapters, fixed
  loss weights.
- **D** (`exp_d_shared_adapters_learned_balance`) -- shared backbone +
  adapters + learned loss balancing.

-- all at a single **`SEED`** you set below (not the 3-seed protocol; use
`notebooks/train_evaluate_colab.ipynb` for the full multi-experiment,
multi-seed pipeline).

**What this notebook is.** An orchestration layer around this
repository's real code -- every step below calls the actual scripts under
`scripts/` exactly as they exist in the repository, streamed live to this
cell's output (unbuffered, per-epoch), never reimplemented here.

**Live progress.** Every training subprocess is run with `-u`
(unbuffered) and streams straight to this notebook's output as it
happens -- one detailed block per epoch (train/val loss, age MAE/RMSE,
gender accuracy/F1, learning rates, early-stopping counter, checkpoint
path; see `src/training/progress.py::format_epoch_report`), not just a
result at the end.

**Resume safety.** Every stage (train / calibrate / evaluate) is checked
and skipped independently based on whether its own artifact already
exists, and the run directory is synced to Google Drive after every
experiment -- so a Colab session timeout never loses completed work. Set
`RESUME_RUN_ID` to a previous run's printed `RUN_ID` to continue it in a
fresh session; the run directory is restored from Drive automatically.

**Ethical framing (kept intact).** The "gender-label" task predicts a
categorical field defined by the source dataset's own documentation -- it
is **not** a determination of a person's gender identity. Research/
education only.

## 2. User configuration

In [ ]:
# ============================================================
# USER CONFIGURATION -- edit this cell, then run the notebook top to bottom.
# ============================================================

SEED = 42  # the single seed these four experiments train at

SMOKE_TEST = False  # True = 1 epoch/3 batches per experiment, fast pipeline check only, NOT scientific results
FORCE_RERUN = False  # True = retrain everything from scratch, ignoring existing checkpoints
ALLOW_TEST_FAILURES = False

# 0b vs D pairwise residual-connection comparison (scripts/compare_backbones.py
# supports 2+ named checkpoints -- SimpleCNN is not required for this pair).
RUN_RESIDUAL_ABLATION = True

MAX_EPOCHS = 40  # ceiling per stage/warm-up; early stopping (below) usually stops sooner
EARLY_STOPPING_PATIENCE = 12

# Optional hard cap on batches-per-epoch, independent of MAX_EPOCHS -- None
# (default) means no limit. SMOKE_TEST overrides this to a small number.
MAX_BATCHES_PER_EPOCH = None

# --- Training-quality settings (configs/training.yaml / configs/model.yaml
# defaults, exposed here for convenience) -- applied identically to every
# experiment, so none of this changes which architecture the results say
# is better; it only improves training quality across the board.
# ------------------------------------------------------------------------
DIFFERENTIAL_LR_ENABLED = True
BACKBONE_LR_MULTIPLIER = 0.1  # 1.0 disables the differential effect (same LR everywhere)
ADAPTER_BOTTLENECK_DIM = 256
LOSS_BALANCING_WARMUP_EPOCHS = 3

REPO_URL = "https://github.com/adischwartz15/AgeGender.git"
REPO_BRANCH = "remove/volo-pretrained-transfer-learning"  # TODO: change back to "main" once this branch is merged

# Set to an existing run's RUN_ID (printed by a previous execution of this
# notebook) to continue that run instead of starting a new one. Leave
# None for a fresh run -- a fresh run NEVER overwrites an existing run
# directory (see the run-ID cell below).
RESUME_RUN_ID = None

USE_GOOGLE_DRIVE = True

# Set this if you'd rather download the dataset via the Kaggle API from
# within Colab instead of using Drive.
KAGGLE_DATASET_SLUG = None

# Set this only if the dataset is already available in Drive.
DRIVE_DATASET_DIR = None

if FORCE_RERUN:
    from IPython.display import Markdown, display
    display(Markdown(
        "> **Full clean rerun enabled**\n>\n"
        "> All reported results in this run are generated from newly trained "
        "models\n> and will not reuse prior checkpoints or metrics."
    ))

In [ ]:
# ============================================================
# Helper library -- shared plumbing used by every phase below.
# None of this reimplements model/dataset/training/evaluation logic; it only
# runs the repository's own scripts, and manages files/manifests around them.
# ============================================================

import datetime
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from IPython.display import Image, Markdown, display


def run_command(cmd, cwd=None, log_path=None, check=True, env=None):
    """Run a subprocess, streaming output live and optionally capturing it to a log file."""
    printable = cmd if isinstance(cmd, str) else " ".join(str(c) for c in cmd)
    print(f"$ {printable}")
    full_env = {**os.environ, **(env or {})}
    proc = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, shell=isinstance(cmd, str), env=full_env,
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    proc.wait()
    if log_path:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text("".join(lines), encoding="utf-8")
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {proc.returncode}): {printable}"
            + (f"\nSee log: {log_path}" if log_path else "")
        )
    return proc.returncode, "".join(lines)


def write_manifest(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(data, fh, indent=2, default=str)
    return path


def load_json(path):
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_copy2(src, dst):
    """shutil.copy2, but skips (with a message) instead of raising when src and dst resolve to the same file.

    This happens on a resumed run whose RUN_DIR already points directly at
    the persistent Drive/output run directory (e.g. a Colab run resumed
    from a Drive-backed RUN_DIR): the log/checkpoint/metrics files under
    RUN_DIR are already being written to their final destination, so a
    later "sync to persistent storage" copy step would otherwise raise
    shutil.SameFileError trying to copy a file onto itself.
    """
    src, dst = Path(src), Path(dst)
    if src.resolve() == dst.resolve():
        print(f"Skipping copy: source and destination are the same file ({dst})")
        return dst
    return shutil.copy2(src, dst)


def copy_tree_merge(src, dst):
    """Copy every file under src into dst (creating dst), without touching unrelated existing dst content."""
    src, dst = Path(src), Path(dst)
    if not src.exists():
        return []
    dst.mkdir(parents=True, exist_ok=True)
    copied = []
    for path in src.rglob("*"):
        if path.is_file():
            target = dst / path.relative_to(src)
            target.parent.mkdir(parents=True, exist_ok=True)
            safe_copy2(path, target)
            copied.append(target)
    return copied


def move_tree_clear(src, dst):
    """Move every file under src into dst, then remove the now-empty src.

    Used for the couple of scripts (generate_gradcam.py, run_robustness.py)
    whose output directory is not configurable per run, so consecutive
    experiments would otherwise collide.
    """
    copied = copy_tree_merge(src, dst)
    if Path(src).exists():
        shutil.rmtree(src)
    return copied


def flatten_overrides(obj, prefix=""):
    """Flatten a nested dict into a flat list of ["--set", "a.b.c=value", ...] CLI args.

    Mirrors src/utils/config.py's own --set parsing (parse_cli_overrides /
    _coerce_scalar) exactly, so every value round-trips the same way it
    would from a hand-typed CLI override.
    """
    args = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            new_prefix = f"{prefix}.{key}" if prefix else key
            args.extend(flatten_overrides(value, new_prefix))
    else:
        value = obj
        if value is None:
            value = "null"
        elif isinstance(value, bool):
            value = "true" if value else "false"
        args += ["--set", f"{prefix}={value}"]
    return args


def display_image(path, caption=None):
    path = Path(path)
    if not path.exists():
        print(f"(plot not available: {path})")
        return
    if caption:
        display(Markdown(f"**{caption}**"))
    display(Image(filename=str(path)))


def artifact_ready(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0


def validate_required_artifacts(paths):
    missing = [str(p) for p in paths if not artifact_ready(p)]
    if missing:
        raise RuntimeError("Required artifact(s) missing or empty:\n  " + "\n  ".join(missing))
    return True


print("Helper library loaded.")

## 3. Runtime and GPU verification

In [ ]:
# ============================================================
# Detect Google Colab
# ============================================================

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Google Colab: {IN_COLAB}")
if not IN_COLAB:
    print(
        "This notebook is designed for Google Colab. It may still run "
        "elsewhere, but paths under /content are Colab-specific; consider "
        "notebooks/train_evaluate_kaggle.ipynb or a local checkout instead."
    )

In [ ]:
# ============================================================
# Runtime and GPU verification
# ============================================================

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU memory (GB): {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
    else:
        print(
            "No GPU detected -- training will run on CPU and will be substantially "
            "slower. Enable a GPU accelerator in this platform's runtime settings "
            "before running the training cells below."
        )
except ImportError:
    print("PyTorch not yet installed -- run the dependency installation cell below first.")

## 4. Repository setup

In [ ]:
# ============================================================
# Repository setup
# ============================================================

REPO_DIR = Path("/content/AgeGender")

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    print(f"Repository already present at {REPO_DIR}; fetching latest {REPO_BRANCH}...")
    run_command(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run_command(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
    run_command(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_DIR)
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_command(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

## 5. Dependency installation

In [ ]:
# ============================================================
# Dependency installation
# ============================================================

run_command(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=REPO_DIR,
)

# Editable install is a convenience for `import src...` directly from notebook
# cells; every script under scripts/ already inserts the repo root onto its
# own sys.path and does not depend on this succeeding. pyproject.toml also
# currently declares requires-python >=3.11, so this is skipped gracefully
# (not treated as fatal) on older interpreters.
try:
    run_command([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO_DIR)
    print("Editable install succeeded.")
except Exception as exc:
    print(f"Editable install skipped ({exc}); continuing with sys.path only (this is expected and fine).")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch  # noqa: E402  (re-import after installation, in case this is a fresh runtime)

print("Dependencies installed.")

## 6. Run ID, run directory, and environment manifest

Local workspace: `/content/agegender_runs/<RUN_ID>`. Persistent Drive destination (when `USE_GOOGLE_DRIVE=True`): `/content/drive/MyDrive/AgeGender/runs/<RUN_ID>`.

In [ ]:
# ============================================================
# Run ID and run directory -- a RESUME_RUN_ID does NOT require the local
# directory to already exist (a fresh Colab VM never has it); it is
# (re)created here and, if USE_GOOGLE_DRIVE=True, restored from Drive in
# the next cell. Restart-safety then comes from the per-stage checkpoint/
# metrics checks below finding what was restored.
# ============================================================

WORKSPACE_DIR = Path("/content/agegender_runs")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)


def _generate_run_id(seed):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
    return f"{timestamp}_c_d_0b_0c_seed{seed}"


if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
    RUN_DIR = WORKSPACE_DIR / RUN_ID
    print(f"Resuming run: {RUN_DIR} (restored from Drive in the next cell, if USE_GOOGLE_DRIVE=True)")
else:
    candidate_id = _generate_run_id(SEED)
    candidate_dir = WORKSPACE_DIR / candidate_id
    suffix = 1
    # Never overwrite an existing run directory silently.
    while candidate_dir.exists():
        suffix += 1
        candidate_dir = WORKSPACE_DIR / f"{candidate_id}_{suffix}"
    RUN_ID, RUN_DIR = candidate_dir.name, candidate_dir
    print(f"Starting new run: {RUN_DIR}")

RUN_SUBDIRS = [
    "config", "logs", "tests", "data_quality", "checkpoints", "metrics", "plots",
    "calibration", "gradcam", "robustness", "knn", "reports", "manifests",
    "archives", "experiments",
]
for _sub in RUN_SUBDIRS:
    (RUN_DIR / _sub).mkdir(parents=True, exist_ok=True)

print(f"RUN_ID  = {RUN_ID}")
print(f"RUN_DIR = {RUN_DIR}")

In [ ]:
# ============================================================
# Mount Google Drive (only if USE_GOOGLE_DRIVE=True) and, when resuming
# an existing RUN_ID, restore whatever was already synced there into this
# fresh runtime's local RUN_DIR -- a new Colab VM never has yesterday's
# /content, only Drive does, so without this step RESUME_RUN_ID would
# silently start every experiment over instead of skipping completed ones.
# ============================================================

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted at /content/drive")

    _drive_run_dir = Path("/content/drive/MyDrive/AgeGender/runs") / RUN_ID
    if _drive_run_dir.exists():
        for _sub in RUN_SUBDIRS:
            _restored = copy_tree_merge(_drive_run_dir / _sub, RUN_DIR / _sub)
            if _restored:
                print(f"Restored {len(_restored)} file(s) into {RUN_DIR / _sub} from Drive.")
    elif RESUME_RUN_ID:
        print(
            f"WARNING: RESUME_RUN_ID='{RESUME_RUN_ID}' was set but no Drive folder "
            f"found at {_drive_run_dir} -- nothing to restore, every experiment will "
            "run from scratch. Check the run ID (see a previous execution's printed "
            "RUN_DIR) if this is unexpected."
        )
else:
    print("Google Drive not mounted (USE_GOOGLE_DRIVE=False or not running in Colab).")


def sync_after_phase(phase_label):
    """Copy the run directory to Drive after a major phase, when enabled."""
    if IN_COLAB and USE_GOOGLE_DRIVE:
        dest = Path("/content/drive/MyDrive/AgeGender/runs") / RUN_ID
        copy_tree_merge(RUN_DIR, dest)
        print(f"[drive sync: {phase_label}] -> {dest}")

In [ ]:
# ============================================================
# Reproducibility and environment manifest
# (never includes credentials -- git status/log and package versions only)
# ============================================================

def _get_git_info(repo_dir):
    try:
        sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip()
        branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=repo_dir, text=True).strip()
        status = subprocess.check_output(["git", "status", "--short"], cwd=repo_dir, text=True)
    except Exception as exc:
        sha, branch, status = f"unavailable ({exc})", "unavailable", ""
    return sha, branch, status


git_sha, git_branch, git_status = _get_git_info(REPO_DIR)

gpu_info = {}
if torch.cuda.is_available():
    gpu_info = {
        "name": torch.cuda.get_device_name(0),
        "total_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
        "cuda_version": torch.version.cuda,
    }

environment_manifest = {
    "run_id": RUN_ID,
    "git_commit_sha": git_sha,
    "git_branch": git_branch,
    "git_status_short": git_status,
    "python_version": sys.version,
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": gpu_info.get("cuda_version"),
    "gpu": gpu_info,
    "platform": platform.platform(),
    "seed": SEED,
    "smoke_test": SMOKE_TEST,
    "force_rerun": FORCE_RERUN,
    "generated_at_utc": datetime.datetime.utcnow().isoformat() + "Z",
}
write_manifest(RUN_DIR / "manifests" / "environment.json", environment_manifest)
print(json.dumps(environment_manifest, indent=2))

In [ ]:
sync_after_phase("repository_setup")

## 7. Dataset setup

Do not copy raw dataset images to Drive by default -- only metadata, split files, configs, logs, checkpoints, metrics, plots, reports, and archives are synchronized.

In [ ]:
# ============================================================
# Kaggle credentials (only needed if KAGGLE_DATASET_SLUG is set below)
# Never printed, logged, or written to a manifest.
# ============================================================

if KAGGLE_DATASET_SLUG and not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        print("Loaded Kaggle credentials from Colab Secrets.")
    except Exception:
        import getpass
        print("Enter Kaggle API credentials (hidden input; never printed or logged):")
        os.environ["KAGGLE_USERNAME"] = getpass.getpass("KAGGLE_USERNAME: ")
        os.environ["KAGGLE_KEY"] = getpass.getpass("KAGGLE_KEY: ")
elif KAGGLE_DATASET_SLUG:
    print("Kaggle credentials already present in the environment.")
else:
    print("KAGGLE_DATASET_SLUG not set; skipping (DRIVE_DATASET_DIR will be used instead).")

In [ ]:
# ============================================================
# Dataset setup
# ============================================================

if DRIVE_DATASET_DIR:
    dest = REPO_DIR / "data" / "raw"
    dest.mkdir(parents=True, exist_ok=True)
    print(f"Using dataset already available in Drive: {DRIVE_DATASET_DIR}")
    copy_tree_merge(DRIVE_DATASET_DIR, dest)
elif KAGGLE_DATASET_SLUG:
    os.environ["KAGGLE_DATASET_SLUG"] = KAGGLE_DATASET_SLUG
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        raise RuntimeError(
            "KAGGLE_USERNAME/KAGGLE_KEY are not set. Run the credentials cell "
            "above (Colab Secrets or hidden prompt) before this cell."
        )
    run_command(
        [sys.executable, "-u", "scripts/download_kaggle_data.py"],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "download_data.log",
    )
else:
    raise RuntimeError(
        "No dataset source configured. Set KAGGLE_DATASET_SLUG (with Kaggle "
        "credentials available) or DRIVE_DATASET_DIR in the configuration "
        "cell before running this notebook."
    )

sync_after_phase("dataset_setup")

## 8. Data validation and deterministic split preparation

In [ ]:
# ============================================================
# Data validation and deterministic split preparation
# ============================================================

split_csv_path = RUN_DIR / "data_quality" / "full_metadata_with_splits.csv"
prepare_overrides = [
    "--set", f"paths.splits_dir={(RUN_DIR / 'data_quality').as_posix()}",
    "--set", f"validation.report_dir={(RUN_DIR / 'data_quality').as_posix()}",
]

if split_csv_path.exists() and not FORCE_RERUN:
    print(f"Reusing existing prepared split at {split_csv_path} (FORCE_RERUN=False).")
else:
    run_command(
        [sys.executable, "-u", "scripts/prepare_data.py"] + prepare_overrides,
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "prepare_data.log",
    )

quality_report = load_json(RUN_DIR / "data_quality" / "data_quality_report.json")
split_hash = sha256_file(split_csv_path)

print("Data quality report:")
print(json.dumps(quality_report, indent=2))
print(f"\nSplit file SHA-256: {split_hash}")

write_manifest(
    RUN_DIR / "manifests" / "data_manifest.json",
    {**quality_report, "split_file_sha256": split_hash, "split_file": str(split_csv_path)},
)
sync_after_phase("data_preparation")

## 9. Automated tests

In [ ]:
# ============================================================
# Automated tests (must pass, or ALLOW_TEST_FAILURES=True, before training)
# ============================================================

pytest_log = RUN_DIR / "tests" / "pytest.log"
junit_path = RUN_DIR / "tests" / "pytest_junit.xml"

returncode, _ = run_command(
    [sys.executable, "-m", "pytest", "-q", f"--junitxml={junit_path}"],
    cwd=REPO_DIR, log_path=pytest_log, check=False,
)
tests_passed = returncode == 0

print(f"\nPython tests: {'PASS' if tests_passed else 'FAIL'}")

if not tests_passed and not ALLOW_TEST_FAILURES:
    raise RuntimeError(
        "Automated tests failed and ALLOW_TEST_FAILURES=False. Fix the failures "
        f"(see {pytest_log}) before running expensive training, or set "
        "ALLOW_TEST_FAILURES=True in the configuration cell to proceed anyway "
        "(not recommended before a reported research run)."
    )

In [ ]:
sync_after_phase("tests")

## 10. Experiment selection (0b, 0c, C, D)

In [ ]:
# ============================================================
# Experiment selection -- only these four, at SEED.
# ============================================================

import yaml

PLAIN_DEEP18_EXPERIMENT = "exp_0b_plain_deep18_no_skip_shared_adapters_learned_balance"
NO_ZERO_INIT_EXPERIMENT = "exp_0c_custom_resnet18_no_zero_init_shared_adapters_learned_balance"
SHARED_ADAPTERS_EXPERIMENT = "exp_c_shared_adapters"
RESNET_EXPERIMENT = "exp_d_shared_adapters_learned_balance"

experiments_cfg = yaml.safe_load((REPO_DIR / "configs" / "experiments.yaml").read_text(encoding="utf-8"))["experiments"]

selected_experiments = [
    PLAIN_DEEP18_EXPERIMENT, NO_ZERO_INIT_EXPERIMENT,
    SHARED_ADAPTERS_EXPERIMENT, RESNET_EXPERIMENT,
]
missing = [name for name in selected_experiments if name not in experiments_cfg]
if missing:
    raise RuntimeError(
        f"Experiment(s) not found in configs/experiments.yaml: {missing}. Stopping rather "
        "than silently running a different set of experiments than requested."
    )
print(f"Will train/evaluate at seed={SEED} (already-complete ones are skipped, not retrained): {selected_experiments}")

if SMOKE_TEST:
    MAX_EPOCHS = min(MAX_EPOCHS, 1)
    EARLY_STOPPING_PATIENCE = min(EARLY_STOPPING_PATIENCE, 1)
    MAX_BATCHES_PER_EPOCH = MAX_BATCHES_PER_EPOCH or 3
    print(
        f"SMOKE_TEST=True: capping MAX_EPOCHS/EARLY_STOPPING_PATIENCE to "
        f"{MAX_EPOCHS}/{EARLY_STOPPING_PATIENCE} and batches-per-epoch to "
        f"{MAX_BATCHES_PER_EPOCH} for a fast integration check only. These "
        "results are NOT final scientific findings."
    )

## 11. Training

Each experiment below is trained, calibrated, evaluated, and synchronized
to Drive in turn -- so a Colab session timeout after experiment *N* still
leaves experiments `1..N` fully persisted in Drive. Re-running this
notebook (same `RUN_ID`, or set `RESUME_RUN_ID`) will skip already-complete
experiments unless `FORCE_RERUN=True`. Per-epoch progress prints live to
this cell's output as each experiment trains (see the overview cell).

In [ ]:
# ============================================================
# Training helpers (orchestration only -- calls scripts/train.py,
# scripts/calibrate.py, scripts/build_knn_index.py, scripts/evaluate.py)
#
# Each stage (train / calibrate / build k-NN index / evaluate) is checked
# and skipped independently, based only on whether *that stage's own*
# artifact already exists. This is what makes the pipeline stage-level
# restart-safe: if evaluation crashes after a successful training run, only
# evaluation reruns next time -- training is never redone just because a
# later stage failed.
# ============================================================

def experiment_paths(run_dir, experiment_name, seed):
    base = run_dir / "experiments" / experiment_name / f"seed_{seed}"
    return {
        "base": base,
        "checkpoint_dir": base / "checkpoints",
        "output_dir": base,
        "calibration_dir": base / "calibration",
        "knn_dir": base / "knn",
        "config_dir": base / "config",
        "log_dir": base / "logs",
    }


def _stage_status(artifact_path, force_rerun):
    exists = artifact_path.exists()
    if exists and not force_rerun:
        return "skipped", f"skipped, found at {artifact_path}"
    if exists and force_rerun:
        return "rerun", f"rerunning (FORCE_RERUN=True), would have reused {artifact_path}"
    return "run", "running, artifact missing"


def print_stage_plan(experiment_name, seed, paths, include_knn):
    """Print the skip/run decision for every stage before executing any of
    them, so a restart's behavior is transparent up front."""
    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    calibration_artifact = paths["calibration_dir"] / "conformal_calibration.json"
    knn_index = paths["knn_dir"] / "knn_baseline.pkl"
    metrics_path = paths["output_dir"] / "metrics" / f"{experiment_name}_test_metrics.json"

    print(f"[{experiment_name} seed={seed}] Stage plan:")
    print(f"  - training:   {_stage_status(checkpoint, FORCE_RERUN)[1]}")
    print(f"  - calibration: {_stage_status(calibration_artifact, FORCE_RERUN)[1]}")
    if include_knn:
        print(f"  - k-NN index: {_stage_status(knn_index, FORCE_RERUN)[1]}")
    else:
        print("  - k-NN index: not requested for this experiment")
    print(f"  - evaluation: {_stage_status(metrics_path, FORCE_RERUN)[1]}")


def train_one_experiment(experiment_name, seed):
    spec = experiments_cfg[experiment_name]
    paths = experiment_paths(RUN_DIR, experiment_name, seed)
    for value in paths.values():
        value.mkdir(parents=True, exist_ok=True)

    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    if checkpoint.exists() and not FORCE_RERUN:
        print(f"[{experiment_name} seed={seed}] training: skipped, checkpoint found ({checkpoint})")
        return paths

    overrides_args = flatten_overrides(spec.get("overrides", {}))
    notebook_overrides = {
        "seed": seed,
        "training": {
            "seed": seed,
            "warm_up_from_scratch": {"epochs": MAX_EPOCHS},
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "max_train_batches_per_epoch": MAX_BATCHES_PER_EPOCH,
            "max_val_batches_per_epoch": MAX_BATCHES_PER_EPOCH,
            "differential_lr": {"enabled": DIFFERENTIAL_LR_ENABLED, "backbone_lr_multiplier": BACKBONE_LR_MULTIPLIER},
        },
        "model": {
            "adapters": {"bottleneck_dim": ADAPTER_BOTTLENECK_DIM},
            "loss_balancing": {"learned_uncertainty": {"warmup_epochs": LOSS_BALANCING_WARMUP_EPOCHS}},
        },
        "paths": {
            "checkpoint_dir": paths["checkpoint_dir"].as_posix(),
            "output_dir": paths["output_dir"].as_posix(),
            "splits_dir": (RUN_DIR / "data_quality").as_posix(),
        },
        "calibration": {"output_dir": paths["calibration_dir"].as_posix()},
        "knn": {"index_dir": paths["knn_dir"].as_posix()},
    }
    overrides_args += flatten_overrides(notebook_overrides)

    write_manifest(
        paths["config_dir"] / "notebook_overrides.json",
        {"experiment_overrides": spec.get("overrides", {}), "notebook_overrides": notebook_overrides},
    )

    print(
        f"[{experiment_name} seed={seed}] training: "
        f"{'rerunning (FORCE_RERUN=True)' if checkpoint.exists() else 'running, checkpoint missing'} "
        f"(max_epochs={MAX_EPOCHS}, patience={EARLY_STOPPING_PATIENCE})"
    )
    run_command(
        [sys.executable, "-u", "scripts/train.py", "--experiment-name", experiment_name] + overrides_args,
        cwd=REPO_DIR, log_path=paths["log_dir"] / "train.log",
    )
    print(f"[{experiment_name} seed={seed}] training: checkpoint written to {checkpoint}")

    # Every checkpoint embeds the full merged config it was produced with
    # (src/training/checkpointing.py:save_checkpoint) -- extract and save it
    # alongside the notebook-level override diff above, so the *actual*
    # resolved configuration is on disk, not just the requested overrides.
    if checkpoint.exists():
        import torch as _torch

        resolved_config = _torch.load(checkpoint, map_location="cpu")["config"]
        write_manifest(paths["config_dir"] / "resolved_config.json", resolved_config)
    return paths


def calibrate_one_experiment(experiment_name, seed, paths):
    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    validate_required_artifacts([checkpoint])

    calibration_artifact = paths["calibration_dir"] / "conformal_calibration.json"
    if calibration_artifact.exists() and not FORCE_RERUN:
        print(f"[{experiment_name} seed={seed}] calibration: skipped, artifact found ({calibration_artifact})")
        return

    print(
        f"[{experiment_name} seed={seed}] calibration: "
        f"{'rerunning (FORCE_RERUN=True)' if calibration_artifact.exists() else 'running, artifact missing'}"
    )
    run_command(
        [sys.executable, "-u", "scripts/calibrate.py", "--checkpoint", str(checkpoint)],
        cwd=REPO_DIR, log_path=paths["log_dir"] / "calibrate.log",
    )
    print(f"[{experiment_name} seed={seed}] calibration: artifact written to {calibration_artifact}")


def build_knn_one_experiment(experiment_name, seed, paths):
    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    knn_index = paths["knn_dir"] / "knn_baseline.pkl"
    if knn_index.exists() and not FORCE_RERUN:
        print(f"[{experiment_name} seed={seed}] k-NN index: skipped, index found ({knn_index})")
        return

    print(
        f"[{experiment_name} seed={seed}] k-NN index: "
        f"{'rerunning (FORCE_RERUN=True)' if knn_index.exists() else 'running, index missing'}"
    )
    run_command(
        [sys.executable, "-u", "scripts/build_knn_index.py", "--checkpoint", str(checkpoint)],
        cwd=REPO_DIR, log_path=paths["log_dir"] / "build_knn_index.log",
    )
    print(f"[{experiment_name} seed={seed}] k-NN index: written to {knn_index}")


def evaluate_one_experiment(experiment_name, seed, paths, include_knn):
    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    knn_index = paths["knn_dir"] / "knn_baseline.pkl"
    metrics_path = paths["output_dir"] / "metrics" / f"{experiment_name}_test_metrics.json"

    if metrics_path.exists() and not FORCE_RERUN:
        print(f"[{experiment_name} seed={seed}] evaluation: skipped, metrics found ({metrics_path})")
        return load_json(metrics_path)

    print(
        f"[{experiment_name} seed={seed}] evaluation: "
        f"{'rerunning (FORCE_RERUN=True)' if metrics_path.exists() else 'running, metrics missing'}"
    )
    eval_args = [
        sys.executable, "-u", "scripts/evaluate.py", "--checkpoint", str(checkpoint),
        "--calibration-dir", str(paths["calibration_dir"]),
        "--output-name", f"{experiment_name}_test_metrics",
    ]
    if include_knn and knn_index.exists():
        eval_args += ["--compare-knn", "--knn-path", str(knn_index)]
    run_command(eval_args, cwd=REPO_DIR, log_path=paths["log_dir"] / "evaluate.log")

    validate_required_artifacts([metrics_path])
    print(f"[{experiment_name} seed={seed}] evaluation: metrics written to {metrics_path}")
    return load_json(metrics_path)


def run_experiment_pipeline(experiment_name, seed, include_knn):
    """Run (or skip, per-stage) train -> calibrate -> [k-NN index] -> evaluate for one experiment/seed."""
    preview_paths = experiment_paths(RUN_DIR, experiment_name, seed)
    for value in preview_paths.values():
        value.mkdir(parents=True, exist_ok=True)
    print_stage_plan(experiment_name, seed, preview_paths, include_knn)

    paths = train_one_experiment(experiment_name, seed)
    calibrate_one_experiment(experiment_name, seed, paths)
    if include_knn:
        build_knn_one_experiment(experiment_name, seed, paths)
    metrics = evaluate_one_experiment(experiment_name, seed, paths, include_knn)
    return paths, metrics


def mirror_to_run_level(paths, flat_name):
    """Copy this experiment/seed's metrics/plots/checkpoints/calibration into the
    flat <RUN_DIR>/{metrics,plots,checkpoints,calibration}/ mirror, renaming
    files from their bare experiment_name prefix to flat_name. flat_name is
    the bare experiment name for the primary seed (so
    src.evaluation.final_report's discovery functions find it directly), or
    f"{experiment_name}_seed{seed}" for additional multi-seed runs (matching
    src.evaluation.comparison's seed-aggregation naming convention).
    """
    experiment_name = paths["base"].parent.name
    for sub in ("metrics", "plots"):
        src_dir = paths["output_dir"] / sub
        if not src_dir.exists():
            continue
        for file_path in src_dir.glob(f"{experiment_name}_*"):
            renamed = flat_name + file_path.name[len(experiment_name):]
            dest = RUN_DIR / sub / renamed
            dest.parent.mkdir(parents=True, exist_ok=True)
            safe_copy2(file_path, dest)
    for checkpoint_path in paths["checkpoint_dir"].glob(f"{experiment_name}_*"):
        renamed = flat_name + checkpoint_path.name[len(experiment_name):]
        dest = RUN_DIR / "checkpoints" / renamed
        dest.parent.mkdir(parents=True, exist_ok=True)
        safe_copy2(checkpoint_path, dest)
    copy_tree_merge(paths["calibration_dir"], RUN_DIR / "calibration" / flat_name)


print("Training helpers ready.")

In [ ]:
# ============================================================
# Training -- runs every selected experiment at SEED. Already-complete
# ones (checkpoint + metrics already on disk) are skipped.
# ============================================================

trained_results = {}
for _i, name in enumerate(selected_experiments, start=1):
    print(f"\n{'=' * 70}\nExperiment {_i}/{len(selected_experiments)}: {name} (seed={SEED})\n{'=' * 70}")
    paths, metrics = run_experiment_pipeline(name, SEED, include_knn=False)
    trained_results[name] = {"paths": paths, "test_metrics": metrics}
    mirror_to_run_level(paths, flat_name=name)

    display_image(RUN_DIR / "plots" / f"{name}_training_curves.png", f"{name}: training curves")
    display_image(RUN_DIR / "plots" / f"{name}_loss_balancing.png", f"{name}: loss balancing")

    param_breakdown_path = RUN_DIR / "metrics" / f"{name}_parameter_breakdown.json"
    param_breakdown = load_json(param_breakdown_path) if param_breakdown_path.exists() else {}
    checkpoint_path = paths["checkpoint_dir"] / f"{name}_best_balanced_score.pt"

    print(f"Checkpoint: {checkpoint_path}")
    print(f"Parameter breakdown: {param_breakdown}")
    print(
        f"[{name}] age_mae={metrics.get('age_mae')} "
        f"gender_accuracy={metrics.get('gender_accuracy')} -- DONE"
    )

    sync_after_phase(f"experiment_{name}")

print(f"\nCompleted {len(trained_results)}/{len(selected_experiments)} experiment(s): {list(trained_results)}")

## 12. Residual-connection ablation (0b vs. D)

`scripts/compare_backbones.py`'s full suite (AURC, paired bootstrap CIs,
tail-error analysis, a final explicitly-conditional interpretation)
between PlainDeep18NoSkip (0b) and Custom ResNet-18 (D) -- the pairwise
comparison that isolates whether residual connections help, holding
depth/width fixed. (SimpleCNN is not required for this pairwise
comparison; `scripts/compare_backbones.py` supports 2 or more named
checkpoints.)

In [ ]:
# ============================================================
# Pairwise backbone comparison: 0b vs. D. Never reimplements
# scripts/compare_backbones.py's analysis here.
# ============================================================

if RUN_RESIDUAL_ABLATION and PLAIN_DEEP18_EXPERIMENT in trained_results and RESNET_EXPERIMENT in trained_results:
    _backbone_short_names = {
        PLAIN_DEEP18_EXPERIMENT: "plain_deep18_no_skip",
        RESNET_EXPERIMENT: "custom_resnet18",
    }
    _checkpoint_args, _calibration_args = [], []
    for _exp_name, _short_name in _backbone_short_names.items():
        _ckpt = trained_results[_exp_name]["paths"]["checkpoint_dir"] / f"{_exp_name}_best_balanced_score.pt"
        _checkpoint_args += ["--checkpoint", f"{_short_name}={_ckpt}"]
        _calibration_args += ["--calibration-dir", f"{_short_name}={trained_results[_exp_name]['paths']['calibration_dir']}"]

    backbone_comparison_dir = RUN_DIR / "reports" / "backbone_comparison"
    run_command(
        [
            sys.executable, "-u", "scripts/compare_backbones.py",
            *_checkpoint_args, *_calibration_args,
            "--resnet-name", "custom_resnet18", "--output-dir", str(backbone_comparison_dir),
        ],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "compare_backbones.log",
    )
    print(f"Backbone comparison artifacts saved to {backbone_comparison_dir}")
    interpretation_path = backbone_comparison_dir / "final_interpretation.md"
    if interpretation_path.exists():
        display(Markdown(interpretation_path.read_text(encoding="utf-8")))
    sync_after_phase("backbone_comparison")
else:
    print(
        "Residual-connection ablation skipped (RUN_RESIDUAL_ABLATION=False, "
        f"or {PLAIN_DEEP18_EXPERIMENT} / {RESNET_EXPERIMENT} did not complete)."
    )

## 13. Final summary

In [ ]:
# ============================================================
# Final summary
# ============================================================

def _build_final_summary_md():
    lines = [f"# Final Summary -- Run {RUN_ID}\n"]
    lines.append("## Platform and hardware\n")
    lines.append(
        f"- Platform: {platform.platform()}\n"
        f"- GPU: {gpu_info if gpu_info else 'CPU only'}\n"
        f"- Python: {sys.version.split()[0]}\n"
        f"- PyTorch: {torch.__version__}\n"
    )
    lines.append("## Git commit\n")
    lines.append(f"- SHA: `{git_sha}`\n- Branch: `{git_branch}`\n")
    lines.append("## Seed and experiments\n")
    lines.append(f"- Seed: {SEED}\n- Experiments: {selected_experiments}\n")
    if SMOKE_TEST:
        lines.append(
            "\n**These are smoke-test results (integration check only) and must "
            "not be treated as final scientific findings.**\n"
        )
    lines.append("## Test status\n")
    lines.append(f"- Python tests: {'PASS' if tests_passed else 'FAIL'}\n")
    lines.append("## Final metrics per experiment\n")
    for name, result in trained_results.items():
        scalar_metrics = {k: v for k, v in result["test_metrics"].items() if not isinstance(v, dict)}
        lines.append(f"### {name}\n\n```\n{json.dumps(scalar_metrics, indent=2)}\n```\n")
    lines.append("## Artifact index\n")
    lines.append(
        f"- Run directory: `{RUN_DIR}`\n"
        "- Environment manifest: `manifests/environment.json`\n"
        "- Checkpoints: `checkpoints/`\n- Metrics: `metrics/`\n- Plots: `plots/`\n"
        "- Calibration: `calibration/`\n"
    )
    if RUN_RESIDUAL_ABLATION:
        lines.append("- Residual-connection ablation (0b vs. D): `reports/backbone_comparison/`\n")
    return "\n".join(lines)


final_summary_md = _build_final_summary_md()
(RUN_DIR / "reports" / "final_summary.md").write_text(final_summary_md, encoding="utf-8")
display(Markdown(final_summary_md))
sync_after_phase("report_generation")

## 14. Artifact persistence and final archive

In [ ]:
# ============================================================
# Final archive
# Archive contains only this run's directory: configs, manifests, logs,
# tests, checkpoints, metrics, plots, reports, calibration. Raw dataset
# images live under REPO_DIR/data/raw, entirely outside RUN_DIR, so they
# are never included.
# ============================================================

archive_base = RUN_DIR.parent / f"{RUN_ID}_archive"
local_archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR)
print(f"Created local archive: {local_archive_path}")

if IN_COLAB and USE_GOOGLE_DRIVE:
    drive_run_dir = Path("/content/drive/MyDrive/AgeGender/runs") / RUN_ID
    copy_tree_merge(RUN_DIR, drive_run_dir)
    drive_archive_path = Path("/content/drive/MyDrive/AgeGender/runs") / Path(local_archive_path).name
    safe_copy2(local_archive_path, drive_archive_path)
    print(f"Synchronized run directory to Google Drive: {drive_run_dir}")
    print(f"Archive copied to Google Drive: {drive_archive_path}")
else:
    print("Google Drive persistence skipped (USE_GOOGLE_DRIVE=False or not running in Colab).")

print("\nRun complete.")
print(f"RUN_ID:  {RUN_ID}")
print(f"RUN_DIR: {RUN_DIR}")